# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing Lane 2: Refresh / Content Opportunity Scoring. The starter dataset (content_refresh_anonymized.csv) has 30,000 content items across 32 clients with trailing-90-day performance metrics, including a is_declining_label column that gives me a real, observed target to score against. I picked this lane over the others because it produces a concrete, ranked action queue that a content team could actually use — deciding which pages to review first — rather than just descriptive signals (Lane 1) or unlabeled clusters (Lane 3).

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Question: Which content pages should be reviewed first for refresh, expansion, protection, or pruning?
Unit of analysis: one row = one content item (page-level, not client-level or day-level).
Decision this improves: which pages a content team puts into their next review sprint, instead of reviewing pages in an arbitrary or manual order.
Who acts on it: a content strategist or editor, who picks the top-ranked pages from the queue and decides whether to refresh, expand, protect, or prune each one.
Cost of a wrong call: a false positive (flagged as declining when it isn't) wastes an editor's time reviewing and possibly rewriting a page that didn't need it. A false negative (a real decline missed) is more costly — a page keeps losing traffic silently for weeks before anyone notices, since nobody is reviewing 30,000 pages by hand every week.
Why data/ML helps: with 32 clients and thousands of pages each, no team can manually check every page regularly. A plain if-statement rule (e.g. "flag any page older than 6 months") would be too blunt — decline patterns likely depend on several signals together (age, position, CTR, engagement) in ways that are hard to hand-write, so a scored model can rank pages more precisely than a single fixed rule, similar to how the example pipeline's model beat the hand-rule baseline (precision@50 of roughly 0.24 → 0.74).
What I will claim: these results are observed and decision-support only — a ranking of risk to help prioritize review, not proof that a refresh will fix a page, and not a claim about how Google's algorithm works.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Total rows:", len(df))
print("Number of clients:", df['client_id'].nunique())

# find the real column name for the decline label
label_cols = [c for c in df.columns if 'declin' in c.lower()]
print("Matching column(s):", label_cols)
print(df.columns.tolist())



Total rows: 30000
Number of clients: 32
Matching column(s): []
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
print(df['trend_direction'].value_counts())
print()
print("Percent declining:", round((df['trend_direction'] == 'declining').mean() * 100, 1), "%")
print("Median content age (days):", df['content_age_days'].median())
print("Percent with avg_position = 0 (no data):", round((df['avg_position'] == 0).mean() * 100, 1), "%")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Percent declining: 0.0 %
Median content age (days): 236.0
Percent with avg_position = 0 (no data): 4.0 %


In [12]:
print("Percent declining (down):", round((df['trend_direction'] == 'down').mean() * 100, 1), "%")

Percent declining (down): 54.2 %


Loading the starter dataset (30,000 pages, 32 clients), I found that 54.2% of pages (16,262 of 30,000) are currently trending down — more than half the inventory shows a real decline signal, which is exactly what a refresh-scoring model needs to learn from. The median content age is 236 days, suggesting staleness is a plausible and common driver of decline across the dataset. However, 4% of pages have avg_position = 0 (no ranking data available), which I'll need to filter or flag separately rather than treat as "worst possible rank."

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This work is observed and decision-support only. The trend_direction label reflects patterns already present in the trailing 90-day data — it does not prove that refreshing a "down" page will cause it to recover, only that it's a reasonable candidate to review first. I cannot claim to have found a causal effect, and I am not predicting or reverse-engineering Google's ranking algorithm — only scoring content against patterns observed in this client set.
A few limits to keep in mind: 4% of pages have avg_position = 0 (no ranking data), so those rows need separate handling rather than being treated as "worst rank." The data covers only 32 clients, so patterns here may not generalize to clients outside this sample. Finally, trend_direction and trend_pct describe the outcome itself, so per the data skill's warning, they can never be used as input features later — only as the target I'm trying to score against.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.